Question 3

In [ ]:
#Part (a)

import cv2 as cv
import numpy as np

# Number of matching points to click
N = 6
n = 0

p1 = np.empty((N, 2), dtype=np.float32)
p2 = np.empty((N, 2), dtype=np.float32)

# Mouse callback function
def draw_circle(event, x, y, flags, param):
    global n
    points = param[0]
    image = param[1]

    if event == cv.EVENT_LBUTTONDOWN:
        cv.circle(image, (x, y), 5, (255, 0, 0), -1)
        points[n] = (x, y)
        n += 1


# Load images
im1 = cv.imread("c1.jpg", cv.IMREAD_REDUCED_COLOR_4)
im2 = cv.imread("c2.jpg", cv.IMREAD_REDUCED_COLOR_4)

if im1 is None or im2 is None:
    print("Error: Images not found. Check the image path.")
    exit()

im1copy = im1.copy()
im2copy = im2.copy()

# Click 6 points on image 1
cv.namedWindow("Image 1")
param = [p1, im1copy]
cv.setMouseCallback("Image 1", draw_circle, param)

while True:
    cv.imshow("Image 1", im1copy)

    if n == N:
        break

    if cv.waitKey(20) & 0xFF == 27:
        break

cv.destroyWindow("Image 1")

# Click same 6 corresponding points on image 2
n = 0
cv.namedWindow("Image 2")
param = [p2, im2copy]
cv.setMouseCallback("Image 2", draw_circle, param)

while True:
    cv.imshow("Image 2", im2copy)

    if n == N:
        break

    if cv.waitKey(20) & 0xFF == 27:
        break

cv.destroyWindow("Image 2")

print("Points from image 1:")
print(p1)

print("Points from image 2:")
print(p2)

# Compute homography
H, mask = cv.findHomography(p1, p2, cv.RANSAC)

print("Homography Matrix:")
print(H)

# Warp image 1 to image 2 perspective
height, width = im2.shape[:2]
warped_im1 = cv.warpPerspective(im1, H, (width, height))

# Display results
cv.imshow("Original Image 1", im1)
cv.imshow("Original Image 2", im2)
cv.imshow("Warped Image 1 to Image 2 Perspective", warped_im1)

cv.waitKey(0)
cv.destroyAllWindows()

Points from image 1:
[[244.  56.]
 [505. 150.]
 [ 78. 536.]
 [319. 632.]
 [290. 356.]
 [341. 189.]]
Points from image 2:
[[323.  23.]
 [550. 181.]
 [379. 177.]
 [286. 321.]
 [242. 604.]
 [ 30. 442.]]
Homography Matrix:
[[-9.60585525e-01 -1.79244684e-01  3.63041959e+02]
 [-5.11151898e-02  1.51572734e-01  1.24307575e+01]
 [-2.46432471e-03 -5.61712362e-04  1.00000000e+00]]


In [ ]:
#Part (b)

# Convert both images to grayscale (important for proper subtraction)
gray_warped = cv.cvtColor(warped_im1, cv.COLOR_BGR2GRAY)
gray_im2 = cv.cvtColor(im2, cv.COLOR_BGR2GRAY)

# Compute absolute difference
diff = cv.absdiff(gray_im2, gray_warped)

# Optional: threshold to highlight clear differences
_, diff_thresh = cv.threshold(diff, 30, 255, cv.THRESH_BINARY)

# Display results
cv.imshow("Difference Image (Gray)", diff)
cv.imshow("Difference Image (Threshold)", diff_thresh)

cv.waitKey(0)
cv.destroyAllWindows()